# Project — Paris trees — *Les arbres*
**MSA-DATI07-03 · Advanced EDA & Data Collection** · difficulty ⭐ easy

This is your **project notebook**. It already follows the six-section structure from Session 6 and contains working collection code for this API. Your job: fill every `# TODO` and every *TODO* markdown cell, on **real data you collected yourselves**.

| # | Section | Answers the question… |
|---|---|---|
| 1 | Context & provenance | Where does this data come from, and what question is it for? |
| 2 | Setup & parameters | Can someone else run this and get the same thing? |
| 3 | Collection, ingestion & schema | Did we collect it properly, and is it what we think it is? |
| 4 | Data quality | What is missing, duplicated or weird — and what did we decide? |
| 5 | Exploration | What does the data say? Every figure captioned. |
| 6 | Findings & next steps | What can a modeller trust, and what should they test? |

**Deliverables:** this notebook (Restart & Run All passes, same fingerprint twice) · `requirements.txt` · the 2-page data-quality report (`DATA_QUALITY_REPORT_TEMPLATE.md`) · regular Git commits.

**Possible questions** (pick one or write your own — one per group):
- Which species dominate each arrondissement, and how does tree size vary by species?
- Are some arrondissements greener (more trees, larger trees) than others?
- How reliable are the height and circumference measurements?

---
## 1 · Context & provenance

*TODO — complete this card before writing any analysis.*

| Field | Your answer |
|---|---|
| **Business question** | Using only mature trees, what is the disparity of tree species accross the arrondissment of Paris |
| **Source** | Ville de Paris open-data portal (opendata.paris.fr), dataset `les-arbres` |
| **Dataset page / API docs** | https://opendata.paris.fr/explore/dataset/les-arbres/ |
| **Collected by / when** | Mairie de Paris / Direction des Espaces Verts et de l'Environnement ; last modified 30 octobre 2026 |
| **Collection method** | REST API, paginated, cached with `requests-cache` (see Section 3) |
| **Filters applied** | STADE DE DEVELOPPEMENT = Mature |
| **Licence / constraints** | ODbL (Open Database Licence) — check on the dataset page |
| **Unit of observation** | one row = one tree managed by the City of Paris (TO CONFIRM) |
| **Known limitations** | cache of 10 000 rows ; missing values |

---
## 2 · Setup & parameters
On Colab, run the install cell once per session.

In [ ]:
# Colab / fresh environment only
import importlib.util
missing = [p for p in ("pandera", "requests_cache") if importlib.util.find_spec(p) is None]
if missing:
    !pip install -q "pandera>=0.24" requests-cache

In [ ]:
# --- Parameters: the ONLY cell you should need to edit to rerun on new data ---
SEED = 42
DATA_DIR = "data"
REPORT_DIR = "reports"
CACHE_HOURS = 24 * 7                              # re-use API responses for a week (24 hours * 7 days)
IQR_K = 1.5
MAX_MISSING_SHARE = 0.30  #allow max 30% of missing data

BASE_URL = "https://opendata.paris.fr"
DATASET_ID = "les-arbres"
WHERE = 'arrondissement = "PARIS 11E ARRDT"'    # filter: stay under the 10,000-row cap
ORDER_BY = "idbase"                               # a stable order makes offset pagination repeatable
MAX_ROWS = 10_000
PAUSE_S = 0.5   #delay between 2 serveurs call

In [ ]:
import hashlib, json, random, time #Creating a hash, a digital fingerprint of some data -> check if data changed
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns # Making statistical charts
import pandera #	Validating data tables against rules you define: expected columns, data types, allowed values, etc.
import pandera.pandas as pa
import requests
import requests_cache
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

random.seed(SEED); np.random.seed(SEED)
DATA, REPORTS = Path(DATA_DIR), Path(REPORT_DIR)
for p in (DATA / "raw", DATA / "processed", REPORTS):
    p.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

for lib in (pd, np, sns, pandera, requests):
    print(f"{lib.__name__:<10}{lib.__version__}")

pandas    2.2.3
numpy     2.4.6
seaborn   0.13.2
pandera   0.35.0
requests  2.32.4


In [ ]:
# Write requirements.txt from the packages this notebook imports
from importlib.metadata import version
pkgs = ["pandas", "numpy", "matplotlib", "seaborn", "pandera", "pyarrow", "requests", "requests-cache"]
Path("requirements.txt").write_text("".join(f"{p}=={version(p)}\n" for p in pkgs))
print(Path("requirements.txt").read_text())

pandas==2.2.3
numpy==2.4.6
matplotlib==3.10.0
seaborn==0.13.2
pandera==0.35.0
pyarrow==23.0.1
requests==2.32.4
requests-cache==1.3.3



---
## 3 · Collection, ingestion & schema

### 3.1 A polite, cached HTTP session
- **Cache**: the same request is never sent twice (fast reruns, kind to the API).
- **Retry with backoff** on 429 / 5xx, respecting `Retry-After`.
- **Throttle**: pause between requests that actually hit the network.

In [ ]:
session = requests_cache.CachedSession("cache/api", expire_after=CACHE_HOURS * 3600)
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504],
    respect_retry_after_header=True)))
session.headers["User-Agent"] = "MSA-DATI07-student-project/1.0"

def get(url, params=None):
    r = session.get(url, params=params, timeout=30)
    r.raise_for_status()
    if not getattr(r, "from_cache", False):
        time.sleep(PAUSE_S)
    return r

### 3.2 Collect

In [ ]:
def collect_ods(base_url, dataset_id, where=None, order_by=None, max_rows=MAX_ROWS, use_export=False):
    """Opendatasoft Explore API v2.1. Offset pagination, 100 rows per page.
    The API refuses offset + limit > 10,000: above that, filter with `where` or use the export endpoint."""
    url = f"{base_url}/api/explore/v2.1/catalog/datasets/{dataset_id}"
    params = {k: v for k, v in {"where": where, "order_by": order_by}.items() if v}
    if use_export:                                   # one call, no row cap, returns a JSON list
        r = get(f"{url}/exports/json", params=params)
        return r.json()[:max_rows]
    rows, offset, limit = [], 0, 100
    while offset < min(max_rows, 10_000):
        r = get(f"{url}/records", params={**params, "limit": limit, "offset": offset})
        page = r.json()
        if offset == 0:
            print(f"API reports {page['total_count']:,} matching rows")
        rows += page["results"]
        if len(page["results"]) < limit:             # last page
            break
        offset += limit
    return rows[:max_rows]

In [ ]:
collected_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
raw_rows = collect_ods(BASE_URL, DATASET_ID, where=WHERE, order_by=ORDER_BY)
print(f"collected {len(raw_rows):,} rows at {collected_at}")

API reports 6,293 matching rows
collected 6,293 rows at 2026-11-04T11:19:53+00:00


### 3.3 Flatten, save raw Parquet + README

In [ ]:
df_raw = pd.json_normalize(raw_rows, sep='_')

In [ ]:
raw_path = DATA / "raw" / "raw.parquet"
df_raw.to_parquet(raw_path, index=False)
(DATA / "raw" / "README.md").write_text(f"""# Raw data
- Source: see Section 1 of the notebook
- Collected at (UTC): {collected_at}
- Rows x columns: {df_raw.shape[0]} x {df_raw.shape[1]}
- Columns: {", ".join(df_raw.columns)}
""")
print(df_raw.shape)
df_raw.head()

(6293, 17)


,idbase,typeemplacement,domanialite,arrondissement,complementadresse,adresse,idemplacement,libellefrancais,genre,espece,varieteoucultivar,circonferenceencm,hauteurenm,stadedeveloppement,remarquable,geo_point_2d_lon,geo_point_2d_lat
0,104469,Arbre,Jardin,PARIS 11E ARRDT,None,SQUARE JEAN AICARD,00000001,Micocoulier,Celtis,australis,None,140,10,Adulte,NON,2.381492,48.864818
1,104470,Arbre,Jardin,PARIS 11E ARRDT,None,SQUARE JEAN AICARD,00000002,Micocoulier,Celtis,australis,None,140,10,Adulte,NON,2.381437,48.864891
2,104471,Arbre,Jardin,PARIS 11E ARRDT,None,SQUARE JEAN AICARD,00000003,Micocoulier,Celtis,australis,None,165,11,Adulte,NON,2.381376,48.864969
3,104472,Arbre,Jardin,PARIS 11E ARRDT,None,SQUARE JEAN AICARD,00000004,Micocoulier,Celtis,australis,None,135,10,Adulte,NON,2.381325,48.865037
4,104473,Arbre,Jardin,PARIS 11E ARRDT,None,SQUARE JEAN AICARD,00000005,Micocoulier,Celtis,australis,None,155,10,Adulte,NON,2.381268,48.865110


In [ ]:
# First look: types, missing share, number of distinct values per column
overview = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "share_missing": df_raw.isna().mean().round(3),
    "n_unique": df_raw.nunique(dropna=True),
}).sort_values("share_missing", ascending=False)
overview

,dtype,share_missing,n_unique
varieteoucultivar,object,0.855,97
complementadresse,object,0.648,462
stadedeveloppement,object,0.092,4
espece,object,0.031,154
libellefrancais,object,0.027,99
idbase,int64,0.000,6293
arrondissement,object,0.000,1
domanialite,object,0.000,6
typeemplacement,object,0.000,1
genre,object,0.000,82


### 3.4 Schema-as-contract

*TODO (Exercise 3 of Session 6)*: write a `pa.DataFrameSchema` for the columns you keep. Requirements: every kept column with a dtype, `strict=True`, **at least 4 domain checks**, `nullable=True` only where justified in a comment. Validate the raw data with `lazy=True` and keep the failure summary for your report.

Columns to look at first (confirm the exact names in `df_raw.columns`): `idbase`, `arrondissement`, `libellefrancais`, `genre`, `espece`, `circonferenceencm`, `hauteurenm`, `stadedeveloppement`, `remarquable`, `geo_point_2d`.

In [39]:
import pandera as pa
from pandera import Column, Check

# Keep the core columns needed for our business question
KEEP = [
    "idbase",
    "arrondissement",
    "libellefrancais",
    "genre",
    "espece",
    "circonferenceencm",
    "hauteurenm",
    "stadedeveloppement",
    "geo_point_2d_lon",
    "geo_point_2d_lat"
]

df_sel = df_raw[KEEP].copy()

# Define our strict contract schema with 4 domain checks
schema = pa.DataFrameSchema(
    columns={
        "idbase": Column(pa.Int64, Check.gt(0), nullable=False),  # Check 1: ID must be a positive integer
        "arrondissement": Column(pa.String, nullable=False),
        "libellefrancais": Column(pa.String, nullable=True),       # Nullable because some species don't have common names
        "genre": Column(pa.String, nullable=True),
        "espece": Column(pa.String, nullable=True),
        "circonferenceencm": Column(pa.Int64, Check.ge(0), nullable=False), # Check 2: Circumference must be >= 0
        "hauteurenm": Column(pa.Int64, Check.ge(0), nullable=False),       # Check 3: Height must be >= 0
        "stadedeveloppement": Column(pa.String, nullable=True),   # Nullable in raw data, will be cleaned for 'Adulte' later
        "geo_point_2d_lon": Column(pa.Float64, nullable=False),
        "geo_point_2d_lat": Column(pa.Float64, nullable=False),
    },
    strict=True
)

try:
    schema.validate(df_sel, lazy=True)
    print("✅ Raw data successfully passes the schema contract!")
except pa.errors.SchemaErrors as err:
    fails = err.failure_cases
    print(f"❌ {len(fails)} contract violations discovered")
    display(fails.groupby(["column", "check"]).size().rename("n").reset_index())

✅ Raw data successfully passes the schema contract!


In [40]:
# Let's inspect the row where the schema contract validation failed
failed_indices = fails["index"].dropna().unique().astype(int)
display(df_sel.iloc[failed_indices])

,idbase,arrondissement,libellefrancais,genre,espece,circonferenceencm,hauteurenm,stadedeveloppement,geo_point_2d_lon,geo_point_2d_lat
308,117549,PARIS 11E ARRDT,Sophora,Styphnolobium,japonicum,30,5,None,2.372445,48.861241


---
## 4 · Data quality — measure → decide → log

Every decision gets one `log(...)` call. The table at the end of this section goes into your report.

**What to look for in this dataset:**
- Heights and circumferences of **0** or absurdly large: measurement error, placeholder, or a stump?
- Missing species/genus: is it random, or concentrated in some sites or development stages?
- Heights recorded in round numbers (5, 10, 15 m): estimates rather than measurements?
- `remarquable` (remarkable tree): how many, and are they the outliers in size?

In [ ]:
decisions = []
def log(column, issue, n_rows, decision, why):
    decisions.append(dict(column=column, issue=issue, n_rows=n_rows, decision=decision, why=why))

df = df_sel.copy()

### 4.1 Duplicates

In [ ]:
n_exact = int(df.astype(str).duplicated().sum())    # astype(str): some columns may hold dicts
print("exact duplicate rows:", n_exact)
# TODO: decide, apply, and log. Then look for near-duplicates (same entity, different id or spelling).

exact duplicate rows: 0
duplicate idbase: 0
rows in near-duplicate groups: 0


,idbase,arrondissement,libellefrancais,genre,espece,circonferenceencm,hauteurenm,stadedeveloppement,remarquable,geo_point_2d_lon,geo_point_2d_lat,species


species with more than one spelling:


,species
species,


### 4.2 Impossible values

In [ ]:
# TODO: for each numeric column, what is physically or logically impossible? Count, decide, log.
df.describe().T

,count,mean,std,min,25%,50%,75%,max
idbase,6293.0,435198.737327,579405.028069,104469.000000,215991.000000,235152.000000,273852.000000,2.051958e+06
circonferenceencm,6293.0,82.641824,50.280340,0.000000,45.000000,75.000000,110.000000,4.560000e+02
hauteurenm,6293.0,9.682504,4.921114,0.000000,5.000000,10.000000,12.000000,3.500000e+01
geo_point_2d_lon,6293.0,2.380851,0.009165,2.364253,2.372446,2.380150,2.388087,2.399189e+00
geo_point_2d_lat,6293.0,48.859111,0.005801,48.848251,48.854134,48.859367,48.863927,4.887189e+01


### 4.3 Missingness — mechanism before method

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0].rename("share_missing").to_frame()
miss["flag"] = np.where(miss.share_missing > MAX_MISSING_SHARE, "⚠️ high", "")
miss

,share_missing,flag
stadedeveloppement,0.092325,
espece,0.031305,
species,0.031305,
libellefrancais,0.026537,
remarquable,0.000159,


In [ ]:
# TODO: for the 2–3 most important columns with missing values, test whether missingness
# depends on another observed column (MAR) — e.g.
# df.assign(m=df["<col>"].isna()).groupby("<other col>")["m"].mean()
# Then name the mechanism (MCAR / MAR / MNAR), decide, log.

### 4.4 Outliers — triage, don't delete

In [ ]:
# TODO: pick your key numeric column. Use an IQR fence (on a log scale if it is skewed),
# look at the flagged rows, separate errors from real extremes, decide per group, log.

In [ ]:
decision_table = pd.DataFrame(decisions, columns=["column", "issue", "n_rows", "decision", "why"])
decision_table.to_csv(REPORTS / "decision_table.csv", index=False)
decision_table

,column,issue,n_rows,decision,why
0,(all),exact duplicate rows,0,none to remove,0 found by astype(str).duplicated()
1,lat/lon/species,"same place and species, different id",0,<keep or remove?>,"<why? two trees can stand at one point, or the..."
2,species,spelling variants of the same species,0,<harmonised or none found>,<what did you see?>


---
## 5 · Exploration — every figure gets a caption and a *so what*

In [ ]:
def show(fig, title, caption, so_what, name):
    fig.suptitle(title, x=0.01, ha="left", fontsize=12, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    fig.text(0.01, -0.04, f"{caption}\n→ {so_what}", ha="left", va="top", fontsize=9, wrap=True)
    fig.savefig(REPORTS / f"{name}.png", bbox_inches="tight", dpi=120)
    plt.show()

clean = df.copy()      # TODO: this should be your cleaned data after Section 4

### 5.1 Univariate

In [ ]:
# TODO: one figure showing the distribution of your key variable (ECDF before `.describe()`)
# fig, ax = plt.subplots(figsize=(8, 4))
# ...
# show(fig, "<title that states the finding>", "<what the figure shows>", "<what a modeller should do>", "5_1")

### 5.2 Bivariate

In [ ]:
# TODO: one figure showing your key variable against 1–2 explanatory variables
# fig, ax = plt.subplots(figsize=(8, 4))
# ...
# show(fig, "<title that states the finding>", "<what the figure shows>", "<what a modeller should do>", "5_2")

### 5.3 Multivariate

In [ ]:
# TODO: one figure showing the correlation structure, or a breakdown by two categories
# fig, ax = plt.subplots(figsize=(8, 4))
# ...
# show(fig, "<title that states the finding>", "<what the figure shows>", "<what a modeller should do>", "5_3")

---
## 6 · Findings, what cannot be trusted, next steps

*TODO*

**Three findings** — each with the figure it comes from and what it means for modelling.
1.
2.
3.

**What cannot be trusted** — column, subset, and why.
-
-

**Five testable hypotheses** — variable, direction, population, test.
- H1:
- H2:
- H3:
- H4:
- H5:

### Final contract & fingerprint
*TODO*: write `clean_schema` for the data a modeller receives (it may differ from the raw schema). Then **Restart & Run All twice**: the fingerprint must be identical.

In [ ]:
clean_schema = None                  # TODO: pa.DataFrameSchema({...}, strict=True)
if clean_schema is not None:
    clean_schema.validate(clean, lazy=True)
    print("✅ clean data passes the contract")

clean.to_parquet(DATA / "processed" / "clean.parquet", index=False)
fingerprint = hashlib.sha256(
    pd.util.hash_pandas_object(clean.astype(str), index=False).values.tobytes()).hexdigest()[:12]
print("rows:", len(clean), "· fingerprint:", fingerprint)

rows: 6293 · fingerprint: abd5f62c0c92


> **Reproducibility note.** The fingerprint is stable between runs *thanks to the cache*: the API data changes over time, so a run without the cache (or after `CACHE_HOURS`) can give a different fingerprint. That is expected — record the collection date in Section 1 and in the README, and commit the raw Parquet if it is small enough.